# 🧭 Your gate, inside the loop: an ontology guardrail in a LangGraph agent
### Student edition · about 45 minutes

A webshop support agent proposes refunds. An **ontology gate** checks each proposal against the
shop's facts and refund policy and answers ALLOW or BLOCK. In this notebook the gate lives
*inside* a LangGraph agent: the agent proposes, the gate decides, a BLOCK goes back to the agent
with the policy messages so it can repair its plan, and the refund is paid only after ALLOW.

That design adds a loop and a write, so we also look at the graph itself before it runs, with
gebra: does the repair loop declare a bound, and is the write outside the loop?

**Where this comes from.** In his AI Engineer World's Fair 2026 talk,
[Why Agentic Systems Need Ontologies](https://ai.engineer/talks/Sir59K8ZDPU-ontologies-for-agentic-systems), Frank Coyle (UC Berkeley School of Information)
argues for ontology checks inside the agent loop. His workshop notebooks,
[compass_ai](https://github.com/frankc/compass_ai), build an ALLOW / BLOCK gate for a refund
agent, and the guardrails notebook ends with two next steps: make the gate a check the agent must
pass before any write, and feed the SHACL messages back to the agent so it can repair its plan.
This notebook builds both. It is an independent follow-up, not affiliated with or endorsed by
Frank Coyle; the names, data and code are new.

**Disclosure.** Parts B to D use [gebra](https://github.com/Gebra-Tech/gebra#readme), a
design-time checker for LangGraph graphs. I maintain gebra.

| Part | Topic | What it adds | Time |
|---|---|---|---|
| A | The gate as a node | ALLOW goes to the write, BLOCK goes back to the agent | 10 min |
| B | The repair loop | violations fed back, a declared bound, an exit to a person | 15 min |
| C | Where the write goes | the refund is paid only after ALLOW | 10 min |
| D | What gebra can't see | two graphs that pass and are still wrong | 5 min |

✏️ cells marked **YOUR TURN** are yours. Replace each `____`, run the cell, and read the ✅ / ❌
checks. Every part ends with a 🔄 sync point, so later parts work even if a blank is still open.

## 0 · Setup

Run once. In Colab this installs the package from GitHub. Locally, `make setup` has already
installed it.

In [ ]:
try:
    import ontology_gate
except ImportError:
    %pip install -q "ontology-gate-langgraph @ git+https://github.com/Gebra-Tech/ontology-gate-langgraph@main"
    import ontology_gate

print("ontology_gate", ontology_gate.__version__)

In [ ]:
import inspect
import tempfile
from pathlib import Path

from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph

from ontology_gate.checks import describe_outcome, run_gebra, write_sidecar
from ontology_gate.gate import gate
from ontology_gate.graph import (
    RefundState,
    build_graph,
    commit_refund,
    gate_node,
    human_review,
    make_agent,
    route_after_gate,
    route_without_cap,
    run_case,
)
from ontology_gate.model import ScriptedModel
from ontology_gate.ontology import RULE_MESSAGES

# Sidecars are written here, never into the repository.
SIDECAR_DIR = Path(tempfile.mkdtemp(prefix="refund-gate-"))

____ = "____"  # a blank: replace it with your answer


def check(label, condition):
    """Print ✅ or ❌ for one expectation. Never raises."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + label)


def show(run):
    """One line per gate decision, then whether the case escalated and what was paid."""
    for number, (decision, violations) in enumerate(run.attempts, start=1):
        first = violations[0] if violations else ""
        print(f"  attempt {number}: {decision:5}  {first}".rstrip())
    paid = [entry["refund_id"] for entry in run.ledger]
    print(f"  escalated: {run.escalated}   paid: {paid}")


def gebra_report(graph, sidecar_text):
    """Write the sidecar text to a temp folder and run gebra's checks on the graph."""
    path = write_sidecar(sidecar_text, tempfile.mkdtemp(dir=SIDECAR_DIR))
    return run_gebra(graph, path)

---
## A · The gate as a node  *(10 min)*

The gate takes one proposed refund in Turtle, merges it with the shop's facts and schema, lets
OWL-RL add what follows (a `Manager` is an `Employee`, the object of `forOrder` is an `Order`),
then validates the result with SHACL. It answers **ALLOW** or **BLOCK**, and a BLOCK carries the
policy messages. It reports only what the proposal breaks, never a problem already in the
knowledge base.

The policy has five rules:

In [ ]:
for number, message in enumerate(RULE_MESSAGES, start=1):
    print(number, message)

In [ ]:
proposals = {
    "the AI agent approves": """
        shop:rf_a a shop:Refund ; shop:forOrder shop:o1002 ; shop:requestedBy shop:lena ;
            shop:approvedBy shop:shopbot ; shop:amount 80 .""",
    "a manager approves": """
        shop:rf_b a shop:Refund ; shop:forOrder shop:o1002 ; shop:requestedBy shop:lena ;
            shop:approvedBy shop:priya ; shop:amount 80 .""",
    "a second refund on o1001": """
        shop:rf_c a shop:Refund ; shop:forOrder shop:o1001 ; shop:requestedBy shop:lena ;
            shop:approvedBy shop:marco ; shop:amount 60 .""",
}
for label, turtle in proposals.items():
    result = gate(turtle)
    print(f"{label:26} {result.decision:5}  {result.violations}")

In the graph the gate is a node, and a **router** reads its decision. `route_after_gate` returns
one of three labels: `allow`, `repair` (a BLOCK with repairs left) or `escalate`. A **path map**
says which node each label leads to. Declaring it is what lets LangGraph, and later gebra, see
every route the graph can take.

In [ ]:
print(inspect.getsource(route_after_gate))

In [ ]:
def build_refund_graph(routes, router=route_after_gate):
    """The refund agent, wired by hand: START -> agent -> gate -> (path map)."""
    workflow = StateGraph(RefundState)
    workflow.add_node("agent", make_agent(ScriptedModel()))
    workflow.add_node("gate", gate_node)
    workflow.add_node("commit_refund", commit_refund)
    workflow.add_node("human_review", human_review)
    workflow.add_edge(START, "agent")
    workflow.add_edge("agent", "gate")
    workflow.add_conditional_edges("gate", router, routes)
    workflow.add_edge("commit_refund", END)
    workflow.add_edge("human_review", END)
    return workflow.compile()

### ✏️ YOUR TURN A: wire the gate

Fill in where `allow` goes (the node that pays the refund) and where `repair` goes (back to the
node that proposes). The nodes are `agent`, `gate`, `commit_refund` and `human_review`.

In [ ]:
ALLOW_TARGET = ____
REPAIR_TARGET = ____

routes_a = {"allow": ALLOW_TARGET, "repair": REPAIR_TARGET, "escalate": "human_review"}
runs_a = {}
try:
    graph_a = build_refund_graph(routes_a)
    for case in ("agent-approves", "self-approval"):
        runs_a[case] = run_case(graph_a, case)
        print(case)
        show(runs_a[case])
except Exception as error:
    print("Not yet:", type(error).__name__, error)

check(
    "agent-approves: BLOCK, then ALLOW after one repair",
    lambda: [decision for decision, _ in runs_a["agent-approves"].attempts] == ["BLOCK", "ALLOW"],
)
check(
    "self-approval: BLOCK, then ALLOW after one repair",
    lambda: [decision for decision, _ in runs_a["self-approval"].attempts] == ["BLOCK", "ALLOW"],
)
check(
    "each case pays exactly one refund",
    lambda: len(runs_a) == 2 and all(len(run.ledger) == 1 for run in runs_a.values()),
)

#### 🔄 Sync point
So everyone continues from the same place, here is the path map used from now on.

In [ ]:
GATE_ROUTES = {"allow": "commit_refund", "repair": "agent", "escalate": "human_review"}
correct_graph = build_refund_graph(GATE_ROUTES)
show(run_case(correct_graph, "agent-approves"))

---
## B · The repair loop  *(15 min)*

On a BLOCK the gate writes its messages to `violations` and adds one to `repair_attempts`, and
the agent reads both on its next turn. That is the second next step from compass_ai, and it makes
the graph a loop. Coyle's talk warns (about 13:13) that agent loops can run on, drift and run up
token costs. The duplicate-refund case shows why a loop needs an exit: order `o1001` already has
refund `rf1`, so no approver can make a second refund valid.

In [ ]:
show(run_case(correct_graph, "duplicate-refund"))

Three BLOCKs, then human review: the router escalates once `repair_attempts` reaches
`MAX_REPAIRS`. Here is a graph without that exit. Every BLOCK goes back to the agent, and
`human_review` is still in the graph but nothing routes to it. Only LangGraph's recursion limit
stops the run.

In [ ]:
no_escalate = build_refund_graph(
    {"allow": "commit_refund", "repair": "agent"}, router=route_without_cap
)
try:
    run_case(no_escalate, "duplicate-refund", recursion_limit=25)
except GraphRecursionError:
    print("GraphRecursionError: the loop ran until the recursion limit (25 steps) stopped it")

gebra reads a graph definition before anything runs; it never calls a node. Two of its checks
matter here:

* **P-01** `graph-well-formed`: every node is reachable from START, and every node has somewhere
  to go.
* **P-02** `termination-witness`: every loop *declares* a bound. A pass records that a bound is
  declared (a witness is present). It does not say the loop terminates, and gebra does not check
  that the declaration is true.

Declarations live in a sidecar file, `gebra.toml`. An empty one declares nothing:

In [ ]:
EMPTY_SIDECAR = 'schema = "gebra-sidecar-v1"\n'
print(describe_outcome(gebra_report(no_escalate, EMPTY_SIDECAR), ids=("P-01", "P-02")))

### ✏️ YOUR TURN B: add the exit and declare the bound

1. Send the `escalate` label to the node that hands the case to a person.
2. Declare the loop's bound. gebra reads `[nodes.<node>.variant]` with a `key` that must be a
   key of `RefundState`, and a `measure` that says, in words, what shrinks. Put it on the node
   that increments the repair counter, and name that counter.

In [ ]:
ESCALATE_TARGET = ____
VARIANT_NODE = ____
VARIANT_KEY = ____

routes_b = {"allow": "commit_refund", "repair": "agent", "escalate": ESCALATE_TARGET}
sidecar_b = f"""schema = "gebra-sidecar-v1"

[nodes.{VARIANT_NODE}.variant]
key = "{VARIANT_KEY}"
measure = "MAX_REPAIRS - repair_attempts: gate adds 1 per BLOCK, the router escalates at 3"
"""
report_b, run_b = None, None
try:
    graph_b = build_refund_graph(routes_b)  # route_after_gate escalates at MAX_REPAIRS
    report_b = gebra_report(graph_b, sidecar_b)
    print(describe_outcome(report_b, ids=("P-01", "P-02")))
    run_b = run_case(graph_b, "duplicate-refund")
    show(run_b)
except Exception as error:
    print("Not yet:", type(error).__name__, error)

check("P-01: every node is reachable from START", lambda: report_b["P-01"] == ("pass", None))
check("P-02: the repair loop declares a bound", lambda: report_b["P-02"] == ("pass", None))
check(
    "duplicate-refund ends at human review with nothing paid",
    lambda: run_b.escalated and run_b.ledger == [],
)

#### 🔄 Sync point
The shared sidecar from now on. It also declares what the write does to the world:
`commit_refund` is `billable` and `irreversible`. Part C uses that.

In [ ]:
SIDECAR = """schema = "gebra-sidecar-v1"

[nodes.gate.variant]
key = "repair_attempts"
measure = "MAX_REPAIRS - repair_attempts: gate adds 1 per BLOCK, the router escalates at 3"

[nodes.commit_refund]
effects = ["billable", "irreversible"]
"""
print(describe_outcome(gebra_report(correct_graph, SIDECAR)))

---
## C · Where the write goes  *(10 min)*

The first next step from compass_ai says the agent must pass the gate *before* any write. Coyle's
talk makes the same point: consequential side effects wait until the result is validated (about
16:15), and a second refund on the same order is one of his examples of what an ontology check
should catch (about 18:59).

Here is what happens when the tool runs first and the gate checks after. In the defect graph,
`execute_refund` sits between the agent and the gate, inside the repair loop.

In [ ]:
defect_graph = build_graph("defect")
for case in ("agent-approves", "duplicate-refund"):
    print(case)
    show(run_case(defect_graph, case))

The gate blocked the duplicate refund three times, and it was paid three times anyway. The
agent-approves refund was paid twice: once for the proposal the gate blocked, once for the
repair.

gebra's **P-06** `effect-safety` looks for a node declared `billable` or `irreversible` that the
graph can run again inside a loop, with no idempotency key or compensation hook declared.

In [ ]:
DEFECT_SIDECAR = SIDECAR.replace("[nodes.commit_refund]", "[nodes.execute_refund]")
print(describe_outcome(gebra_report(defect_graph, DEFECT_SIDECAR), ids=("P-02", "P-06")))

### ✏️ YOUR TURN C: pay only after ALLOW

Rewire the graph so the agent goes straight to the gate, and the refund is paid only on `allow`.

In [ ]:
AFTER_AGENT = ____
ON_ALLOW = ____


def build_c():
    workflow = StateGraph(RefundState)
    workflow.add_node("agent", make_agent(ScriptedModel()))
    workflow.add_node("gate", gate_node)
    workflow.add_node("commit_refund", commit_refund)
    workflow.add_node("human_review", human_review)
    workflow.add_edge(START, "agent")
    workflow.add_edge("agent", AFTER_AGENT)
    workflow.add_conditional_edges(
        "gate",
        route_after_gate,
        {"allow": ON_ALLOW, "repair": "agent", "escalate": "human_review"},
    )
    workflow.add_edge("commit_refund", END)
    workflow.add_edge("human_review", END)
    return workflow.compile()


report_c, runs_c = None, []
try:
    graph_c = build_c()
    report_c = gebra_report(graph_c, SIDECAR)
    print(describe_outcome(report_c, ids=("P-06",)))
    runs_c = [run_case(graph_c, case) for case in ("agent-approves", "duplicate-refund")]
except Exception as error:
    print("Not yet:", type(error).__name__, error)

check(
    "P-06: the billable, irreversible write is outside the loop",
    lambda: report_c["P-06"] == ("pass", None),
)
check("agent-approves is paid once", lambda: len(runs_c[0].ledger) == 1)
check("duplicate-refund is never paid", lambda: runs_c[1].ledger == [] and runs_c[1].escalated)

#### 🔄 Sync point
The correct graph, as the package builds it, is the one you just wired.

There is another way to clear P-06: keep the write in the loop and declare an **idempotency
key**, a state key the write reads and the payment provider deduplicates on. gebra accepts the
declaration when the node declares that it reads the key. It trusts the declaration; it cannot
see whether the provider deduplicates. Run the defect graph again and the ledger still shows
three payments, because nothing in this code deduplicates. Moving the write after ALLOW follows
the rule from the talk instead: no side effect until the result is validated.

In [ ]:
final_graph = build_graph("correct")
print(describe_outcome(gebra_report(final_graph, SIDECAR), ids=("P-06",)))

IDEMPOTENT_SIDECAR = (
    DEFECT_SIDECAR
    + """reads = ["case", "order_id", "proposal"]
idempotent = { key = "order_id" }
"""
)
print()
print("defect graph, with a declared idempotency key:")
print(describe_outcome(gebra_report(defect_graph, IDEMPOTENT_SIDECAR), ids=("P-06",)))
show(run_case(defect_graph, "duplicate-refund"))

---
## D · What gebra can't see  *(5 min)*

Two graphs that pass the gebra checks used here and are still wrong. No blanks in this part.

**A bound that is declared but not enforced.** The `uncapped` graph keeps the sidecar's loop
variant, but its router ignores `MAX_REPAIRS`.

In [ ]:
uncapped = build_graph("uncapped")
print(describe_outcome(gebra_report(uncapped, SIDECAR)))
try:
    run_case(uncapped, "duplicate-refund", recursion_limit=25)
except GraphRecursionError:
    print("at run time: GraphRecursionError, the loop never escalates")

P-02 passes because a bound is declared, and gebra trusts declarations. The declaration is now
false. A P-02 pass records a witness; it is not a statement that the loop ends.

**A write that skips the gate.** The `bypass` graph adds a shortcut: refunds under 50 go from the
agent straight to `commit_refund`.

In [ ]:
bypass = build_graph("bypass")
print(describe_outcome(gebra_report(bypass, SIDECAR)))
show(run_case(bypass, "small-refund"))

Omar's refund of 30 was approved by the AI agent, and the gate never saw it. gebra passes:
`commit_refund` is outside every loop, so P-06 has nothing to report, and gebra does not check
that every write goes through the gate.

What gebra reads is the graph's structure and what the sidecar declares. It does not check the
ontology, the SHACL shapes or any business rule: that is the gate's job, at run time. It reads
LangGraph graphs only, so a plain tool-use loop has to be written as a graph first.

---
### 🎯 Take-aways

| Layer | Question it answers |
|---|---|
| **Pydantic** | Is the input well-formed? |
| **RDFS / OWL** | What follows from the facts? |
| **SHACL gate** | Does this action break policy, at run time? |
| **gebra** | Is the loop bounded and is the write outside it, before the run? |

The ontology decides whether one refund is allowed; the shape of the graph decides how many
times it can be paid. Check both.